# Week 2: Pandas Data Wrangling
## Notebook 07: Memory Optimization: Categoricals, Downcasting & Chunking

### 1. Conceptual Intuition & Real-World Use Case
In enterprise environments, data engineers often encounter a dreaded error: `MemoryError: Unable to allocate array`.
- By default, Pandas loads integers as 64-bit (`int64`, 8 bytes) and floats as 64-bit (`float64`, 8 bytes).
- Text strings are stored as Python `object` references, consuming up to 50+ bytes per string cell!
- A 2GB raw CSV file can expand to **8GB - 12GB of RAM** once loaded into a naive DataFrame.

Through systematic **Downcasting**, **Category Dtypes**, and **Chunking**, we can compress DataFrame RAM usage by **70% to 90%** with zero loss of numerical precision!

---

### 2. Mathematical Foundations: Data Types & RAM Footprint
| Data Type | Values Range | Bytes per Cell |
|---|---|---|
| `int8` | -128 to 127 | 1 byte |
| `int16` | -32,768 to 32,767 | 2 bytes |
| `int32` | -2.1B to +2.1B | 4 bytes |
| `int64` (default) | $\pm 9.22 \times 10^{18}$ | 8 bytes |
| `float32` | Single Precision (7 digits) | 4 bytes |
| `float64` (default) | Double Precision (15 digits) | 8 bytes |
| `category` | Integer Code Lookup Table | 1-2 bytes per row |

In [ ]:
import numpy as np
import pandas as pd

# Step 1: Generate a Large Synthetic DataFrame (100,000 rows)
N = 100000
np.random.seed(42)

df_naive = pd.DataFrame({
    "user_id": np.random.randint(1, 50000, size=N),            # int64
    "age": np.random.randint(18, 90, size=N),                   # int64 (Max value 90 fits in int8!)
    "rating": np.random.uniform(1.0, 5.0, size=N),              # float64 (3 decimals fits in float32!)
    "country": np.random.choice(["USA", "UK", "Canada", "Germany", "France"], size=N), # object strings
    "device": np.random.choice(["iOS", "Android", "Web"], size=N)                       # object strings
})

initial_memory = df_naive.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"Initial Naive DataFrame RAM: {initial_memory:.2f} MB")
print("Data Types Before Optimization:\n", df_naive.dtypes)

# Step 2: Implement Systematic Memory Downcasting
df_opt = df_naive.copy()

# A. Downcast Integers: 'age' (18-90) fits in int8 (-128 to 127)
df_opt["age"] = pd.to_numeric(df_opt["age"], downcast="integer")
df_opt["user_id"] = pd.to_numeric(df_opt["user_id"], downcast="integer")

# B. Downcast Floats: float64 -> float32
df_opt["rating"] = pd.to_numeric(df_opt["rating"], downcast="float")

# C. Convert Low-Cardinality Strings to Categorical
df_opt["country"] = df_opt["country"].astype("category")
df_opt["device"] = df_opt["device"].astype("category")

optimized_memory = df_opt.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"\nOptimized DataFrame RAM: {optimized_memory:.2f} MB")
print("Data Types After Optimization:\n", df_opt.dtypes)

savings = (initial_memory - optimized_memory) / initial_memory * 100
print(f"\nTotal RAM Reduction: {savings:.1f}% Savings!")

# Step 3: Low-Memory Chunked Processing Pattern for 10GB+ CSVs
# Demonstrating chunksize iterator pattern
chunk_accumulator = 0
for chunk in [df_opt.iloc[i : i + 25000] for i in range(0, N, 25000)]:
    # Process each chunk independently in memory
    chunk_accumulator += chunk["rating"].sum()

print(f"Chunked Aggregation Result: Total rating sum = {chunk_accumulator:.2f}")

### 3. Step-by-Step Code Walkthrough
1. **`memory_usage(deep=True)`:** Must specify `deep=True` to inspect actual memory consumed by Python string pointers.
2. **Numeric Downcasting:** `pd.to_numeric(downcast='integer')` automatically inspected max values and selected `int8` for age and `int32` for user IDs.
3. **Categorical Lookup Table:** The strings "USA", "UK", etc. are stored only once in a dictionary table; the column stores 1-byte integer pointers, slashing string memory by over 80%.

---

### 4. Industry Applications & Trade-offs
- **Edge Deployment & Kaggle Competitions:** Maximizing RAM efficiency when training on memory-constrained GPU/TPU instances.
- **Trade-off:** Appending a brand-new unseen category to a Categorical column throws a ValueError unless added to `cat.categories` first.

---

### 5. Top Interview Questions & Answers
**Q1: Why does `df.info()` underestimate memory usage unless `memory_usage='deep'` is set?**
> **Answer:** `object` columns in Pandas contain 64-bit pointers to Python heap string objects. By default, Pandas only measures the memory of the pointers (8 bytes per row) rather than the actual variable-length strings on the heap. `deep=True` traverses the pointers to compute real memory.

**Q2: How does categorical encoding in Pandas save memory?**
> **Answer:** If a column with 1,000,000 rows contains only 5 unique strings (e.g. status codes), storing them as Python strings requires 1,000,000 heap objects. Categorical encoding creates a dictionary of the 5 strings and stores an array of 8-bit integer codes ($0, 1, 2, 3, 4$), reducing memory from $\sim 50$MB to $\sim 1$MB.